# 非線形バネへのC/GMRESの適用

アルゴリズム理解のため非線形バネにたいしてC/GMRESを適用する。

## 1質点モデル

<img src ="images/one-degree-nonlinear-spring.png" style="width:30%;"/>

### 運動方程式

バネの復元力 $F(x)$ は次のように非線形性を持つとする。

$$
F(x) = kx(t) + k_3 x(t)^3
$$

$x(t)$ の方向に力 $u(t)$ を質点 $m$ へ加えた場合、次の運動方程式になる。

$$
m\ddot{x}(t) = u(t) - k(x) - k_3 x(t)^3
$$

### 状態方程式

状態 $X(t)$ を以下のように定義する。

$$
X(t) = [x(t), v(t)]^T
$$

運動方程式を以下のようにまとめる。

$$
\begin{aligned}
\dot{x}(t) &= v(t) \\
\dot{v}(t) &= \frac{1}{m}\left( u(t) - k x(t) - k_3 x(t)^3 \right) 
\end{aligned}
$$

ここから、状態方程式 $\dot{X} = f(X, u)$ は次のようになる。

$$
\frac{d}{dt}
\begin{bmatrix}
x(t) \\ v(t)
\end{bmatrix}
= \begin{bmatrix}
v(t) \\ \dfrac{1}{m}\left( u(t) - k x(t) - k_3 x(t)^3 \right)
\end{bmatrix}
$$

### 拡大評価関数とコスト

拡大評価関数 $\bar{J}$ は次の式である。 

$$
\bar{J} = \Phi(X(T)) + \int_0^T \left[ L(X,u) + \lambda^T \ \left( f(X,u) - \dot{X} \right) \right] dt
$$

位置の目標値を$x_{ref}$ とするように、ランニングコスト$L(X,u)$と終端コスト$\Phi(X(T))$ を設定する。


##### ランニングコスト $L$

ランニングコスト $L(X,u)$ は次のように設定する。

$$
L(X,u) = \frac{1}{2} q_x(x(t) - x_{ref})^2 + \frac{1}{2} q_v v(t)^2 + \frac{1}{2} r u(t)^2
$$

$\bar{J}$の最小化問題になるため、ランニングコスト $L(X,u)$ のそれぞれの意味は次のようになる。

- 第1項目: 位置の目標値との誤差を小さくする
- 第2項目: 速度を大きくしない
- 第3項目: 入力を大きくしない

##### 終端コスト $\Phi$

終端コスト $\Phi(X(T))$ は次のように設定する。

$$
\Phi(X(T)) = \frac{1}{2}q_{xT}(x(T) - x_{ref})^2 + \frac{1}{2}q_{vT}v(T)^2
$$

$\bar{J}$の最小化問題になるため、終端コスト $\Phi(X(T))$ のそれぞれの意味は次のようになる。

- 第1項目: 時刻$T$ における位置と目標位置との誤差を小さくする
- 第2項目: 時刻$T$ における速度を大きくしない

##### 随伴変数　$\lambda$

$\lambda$は状態方程式制約$f(X,u) - \dot{X}$ と内積をとり、$\lambda^T (f(X,u) - \dot{X}) $がスカラーとなるように設定する。状態$X$は2変数であるため、$\lambda$を次のように設定する。

$$
\lambda = [\lambda_x, \lambda_v]^T
$$


### PMP 条件

Hamiltonian $H$ は次の式である。

$$
H(X,u,\lambda) = L(X, u) + \lambda^T f(X, u)
$$

具体的に$H$を構成すると次のようになる。

$$
H = \frac{1}{2} q_x(x(t) - x_{ref})^2 + \frac{1}{2} q_v v(t)^2 + \frac{1}{2} r u(t)^2 + \lambda_x v(t) + \frac{\lambda_v}{m}\left(u(t) - k x(t) - k_3 x(t)^3  \right)
$$

PMP条件を構成するため、$H$ を$X,u$で偏微分、$\Phi$を$X(T)$ で偏微分を行う。また、$H_\lambda = f(X,u)$ である。

$$
\begin{aligned}
H_X &= \frac{\partial H}{\partial X} = \begin{bmatrix} \partial H / \partial x \\ \partial H / \partial v \end{bmatrix} =
\begin{bmatrix}
q_x(x(t) - x_{ref}) - \lambda_v/m \ \left( k + 3 k_3 x(t)^2 \right) \\
q_v v(t) + \lambda_x 
\end{bmatrix} \\
H_u &= \frac{\partial H}{\partial u} = r u(t) + \frac{\lambda_v}{m} \\
\Phi_X &= \frac{\partial \Phi}{\partial X} = \begin{bmatrix} \partial \Phi / \partial x \\ \partial \Phi / \partial v \end{bmatrix} =
\begin{bmatrix}
q_{xT}(x(T) - x_{ref}) \\
q_{vT} v(T)
\end{bmatrix} 
\end{aligned}
$$



$H$を用いて、PMP条件は次のようになる。

$$
\begin{array}{l}
\dot{X} = H_\lambda \\
\dot{\lambda} = - H_X \\
H_u = 0 \\
\lambda(T) = \Phi_X(X(T)) \\
\end{array}
$$ 

よって1質点の非線形バネモデルにおけるPMP条件の式は以下となる。

状態方程式

$$
\boxed{
\frac{d}{dt}
\begin{bmatrix}
x(t) \\ v(t)
\end{bmatrix}
= \begin{bmatrix}
v(t) \\ \dfrac{1}{m}\left( u(t) - k x(t) - k_3 x(t)^3 \right)
\end{bmatrix}
}
$$

随伴方程式

$$
\boxed{
\frac{d}{dt}
\begin{bmatrix}
\lambda_x \\ \lambda_v
\end{bmatrix}
=-\begin{bmatrix}
q_x(x(t) - x_{ref}) - \lambda_v/m \ \left( k + 3 k_3 x(t)^2 \right) \\
q_v v(t) + \lambda_x 
\end{bmatrix}
}
$$

停留条件

$$
\boxed{
r u(t) + \frac{\lambda_v}{m} = 0
}
$$

終端条件

$$
\boxed{
\begin{bmatrix}
\lambda_x(T) \\ \lambda_v(T)
\end{bmatrix} =
\begin{bmatrix}
q_{xT}(x(T) - x_{ref}) \\
q_{vT} v(T)
\end{bmatrix} 
}
$$

### 状態 $X$ 随伴変数 $\lambda$ の時系列の計算式と停留条件式 $F$ の構成

制御入力列 $U(t_k)$ 、状態の初期値 $X[0]$、予測ホライゾンの長さ$T(t_k)=T_f(1 - \exp(-\alpha t_k))$ の$t_k$を設定値として時系列を計算する。

予測ホライゾンのステップ幅 $h(t_k)$ は $T(t_k)$の分割数 $N$ より以下のようになる。

$$
h(t_k) = T(t_k) / N
$$

以下の式は上から順に、状態$X[n]$、終端状態 $\lambda[N]$、随伴変数 $\lambda[n]$ に関する。

$$
\begin{aligned}
\begin{bmatrix}
x[n+1] \\ v[n+1]
\end{bmatrix} &=
\begin{bmatrix}
x[n] \\ v[n]
\end{bmatrix} +
h\begin{bmatrix}
v[n] \\ \dfrac{1}{m}\left( u_n(t_k) - k x[n] - k_3 x[n]^3 \right)
\end{bmatrix}, \quad n = 0, \cdots , N-1 \\
\begin{bmatrix}
\lambda_x[N] \\ \lambda_v[N]
\end{bmatrix} &=
\begin{bmatrix}
q_{xT}(x[N] - x_{ref}) \\
q_{vT} v[N]
\end{bmatrix} \\
\begin{bmatrix}
\lambda_x[n] \\ \lambda_v[n]
\end{bmatrix} &=\begin{bmatrix}
\lambda_x[n+1] \\ \lambda_v[n+1]
\end{bmatrix}
+h\begin{bmatrix}
q_x(x[n] - x_{ref}) - \lambda_v[n+1]/m \ \left( k + 3 k_3 x[n]^2 \right) \\
q_v v[n] + \lambda_x[n+1] 
\end{bmatrix} , \quad n = N-1, \cdots , 1
\end{aligned}
$$

上記を用いて、停留条件式 $F(X(t_k), U(t_k))$ を以下のように構成する。

$$
F(X(t_k), U(t_k)) = 
\begin{bmatrix}
r u_0(t) + \dfrac{\lambda_v[1]}{m} \\
\vdots \\
r u_{N-1}(t) + \dfrac{\lambda_v[N]}{m} \\
\end{bmatrix} = 0
$$

また、この時系列から$F$を構成する計算はC/GMRESの実行中に何度も行われる。そのため

$$
F(U(t_k), X(t_k), t_k)
$$

を関数として定義する。これは、$U(t_k)$ に制御入力列、$X(t_k)$ に実時間 $t_k$ における現在状態を与え、それを予測区間の初期状態$X[0]$として使用する、$t_k$を$T(t_k)$ を計算する入力として与えれれば、それをもとに停留条件の式 $F(U(t_k), X(t_k), t_k)$ を構成する、という意味である。

### 初期制御入力 $u(0)$ の計算

$T(0)=0$ として解析的に $u(0)$ を求める。

停留条件を$T(0)=0$ で構成すると以下の式となる。

$$
r u(0) + \frac{q_{vT} v(0)}{m} = 0 \rightarrow u_0 = -\frac{q_{vT}v(0)}{r\ m} 
$$

ここから、初期状態 $X(0)$ を取得すれば、$u(0)$が計算できるといえる。

$(x(0) - x_{ref})$ の状態を考慮していないが、$u(0)$は求まる。これは $u(t)$ は以下の加速度側の運動方程式にあり、

$$
\dot{v} = \frac{1}{m}\left( u(t) - k x(t) - k_3 x(t)^3 \right)
$$

かつ、Hamiltonian $H$ は以下であるため、

$$
H = L + \lambda_x \dot{x} + \lambda_v \dot{v}
$$

$H$ の $u$ による偏微分は以下のようになり、$u(t)$ に関しては $\lambda_v$ のみが関係するためである。

$$
H_u = r u(t) + \frac{\lambda_v}{m}
$$

$H_u$ には $x$ に対応する $\lambda_x$ が直接出てこない。これは

$$
u \rightarrow \dot{v} \rightarrow v \rightarrow \dot{x} \rightarrow x
$$

という構造になっているためである。

$T(0)=0$ では予測区間が存在しないため、入力$u$によって速度$v$が変化し、その速度変化がさに将来の位置変化へ伝わる過程を予測する時間がない。そのため、初期$u(0)$を計算する問題では位置誤差 $x(0) - x_{ref}$ が $H_u$ に直接現れない。

$T(t)$が$0$から有限長へ伸びると、予測区間内の状態・随伴変数の時系列が計算され、位置誤差の影響が随伴変数を介して停留条件 $H_u$ に反映される。


ここまでがC/GMRESを計算するために必要な設定である。

これらの計算を用いてC/GMRESの制御ループを以下のように実行する。

# C/GMRESの計算

以下のStep 1 から Step 8を繰り返す。

### Step 1 状態取得

制御対象の状態 $X(t_k)$ を取得する。

### Step 2 制御入力列の更新

制御入力列 $U(t_k) \ \in \mathbb{R}^{N}$ は前回のC/GMRESで計算された値を用いる。

$t_0$ の場合、$X(t_0)$ を用いて $u(0)$ を計算し、$U(t_0) \ \in \mathbb{R}^{N}$の制御入力列を構成する。

$$
U(t_0) = [u(0), \cdots , u(0)]^T
$$

### Step 3 制御対象へ制御入力を出力

制御対象へ 制御入力列 $U(t_k)$ の第一要素 $u_0(t_k)$を出力する。

### Step 4 状態$\dot{X}$の計算

$\dot{X}(t_k) = f(X(t_k), u_0(t_k))$ を計算する。

今回の場合、以下の状態方程式を計算する。

$$
\frac{d}{dt}
\begin{bmatrix}
x(t_k) \\ v(t_k)
\end{bmatrix}
= \begin{bmatrix}
v(t_k) \\ \dfrac{1}{m}\left( u_0(t_k) - k x(t_k) - k_3 x(t_k)^3 \right)
\end{bmatrix}
$$

### Step 5 停留条件の計算

$F(U(t_k), X(t_k), t_k)$ を計算する。

### Step 6 右辺 $b_k$ の計算

GMRESの右辺に渡す $b_k$ を計算する。

$F(U(t_k) , X(t_k) + \varepsilon \dot{X}(t_k) , t_k + \varepsilon)$ を計算する。

$$
b_k = - \frac{F(U(t_k) , X(t_k) + \varepsilon \dot{X}(t_k) , t_k + \varepsilon) - F(U(t_k), X(t_k), t_k)}{\varepsilon} - \zeta F(U(t_k), X(t_k), t_k)
$$

### Step 7 GMRESの計算

#### Step 7-1 初期残差の計算

初期残差 $r_0$ の計算に用いる初期解を $\dot{U}^{(0)}(t_k)$を前回の制御ループで計算した $\dot{U}(t_{k-1})$ とする。

$t_0$ の場合、$\dot{U}^{(0)}(t_0)=0$ とする。

$t_k$ の場合、$r_0$ を計算する $F_U \dot{U}^{(0)}(t_k)$ を以下で計算する。

$$
F_U \dot{U}^{(0)}(t_k) = \frac{F(U(t_k) + \varepsilon \dot{U}^{(0)}(t_{k}), X(t_k), t_k) - F(U(t_k), X(t_k), t_k)}{ \varepsilon}
$$

以上より、初期残差を計算する。

$$
r_0 = b_k - F_U \dot{U}^{(0)}(t_k)
$$

#### Step 7-2 直交基底 $v_1$ の計算

$$
v_1 = r_ 0 / \beta , \quad \beta = ||r_0||
$$

#### Step 7-3 Arnoldi法 Step m (m=$1, \cdots $)

$$
\begin{aligned}
F_U(t_k) v_m &= \frac{F(U(t_k) + \varepsilon v_m, X(t_k), t_k) - F(U(t_k), X(t_k), t_k)}{\varepsilon} \\
h_{i, m} &= v_i^T F_U(t_k) v_m \quad (i = 1, \cdots , m) \\
w &= F_U(t_k) v_m - \sum_{i=1}^{m} h_{i,m} v_i \\
h_{m+1, m} &= ||w|| \\
v_{m+1} &= w / h_{m+1, m}
\end{aligned}
$$

この後、これまでのGivens回転を今回構成される 上ヘッセンベルグ行列の列へ適用し、上三角行列を作る。

残差 $g$ が目標よりも小さければ 最小二乗問題を解き、$Y_m$ を計算し、Arnoldi法の繰り返しを終了する。
そうでなければ、Arnoldi法のStep を繰り返す。

#### Step 7-4 $\dot{U}(t_k)$ の近似解を計算

Arnoldi法が停止した Step m では Krylov部分空間の直交基底 $V_m$ と、$Y_m$ が求まっている。
これを用いて、$\dot{U}(t_k)$ を次のように構成する。

$$
\dot{U}(t_k) = \dot{U}^{(0)}(t_k) + V_m Y_m
$$

### Step 8 次の制御入力列 $U(t_{k+1})$ を計算

求めた $\dot{U}(t_k)$ から、次の時刻の $U(t_{k+1})$ を以下の方法で計算する。

$$
U(t_{k+1}) = U(t_k) + \Delta t \  \dot{U}(t_k)
$$


# 実装

PythonによるC/GMRESの実装を行う

In [ ]:
from dataclasses import dataclass
import numpy as np

@dataclass
class Parameters:    
    # 非線形バネモデル
    m: float = 1.0  # 質量
    k: float = 1.0  # バネ定数
    k3: float = 2.0  # 3次のバネ定数
    
    # 目標位置
    x_target: float = 1.0  # 目標位置[m]
    
    # ランニングコストの重み
    qx: float = 1.0  # 状態 位置の重み
    qv: float = 1.0  # 状態 速度の重み
    r: float = 0.1  # 制御入力の重み
    
    # 終端コストの重み
    qxt: float = 10.0  # 終端状態 位置の重み
    qvt: float = 10.0  # 終端状態 速度の重み
    
    # 予測ホライゾン関連
    Tf: float = 5.0  # 予測ホライゾンの最終長さ
    Ta: float = 0.1  # 予測ホライゾンの収束の速さ(大きいほど早く収束する)
    N: int = 50  # 予測ホライゾンの分割数

    # 有限差分の計算項目
    eps: float = 1e-3 # 

    # 計算の安定性に関する項目
    zeta: float = 10.0 # 停留条件への追従

# 運動方程式

class C_GMRES:
    def __init__(self, params: Parameters):
        self.params = params
        self.prev_U = None
        self.prev_dU = None

    # 初期制御入力の計算
    def initial_control_input(self, X0):
        # 初期制御入力をゼロに設定
        U = np.zeros(self.params.N)
        u0 = -self.params.qvt*X0[1]/(self.params.r * self.params.m)  # 初期制御入力の計算
        for i in range(self.params.N):
            U[i] = u0
        return U

    # 状態方程式
    def state_equation(self, X, u):
        m = self.params.m  # 質量
        k = self.params.k  # バネ定数
        k3 = self.params.k3  # 3次のバネ定数
        dx = X[1]
        dv = (u - k * X[0] - k3 * X[0]**3) / m
        return [dx, dv]
    
    # PMP 停留条件式
    def calc_F(self, U, X, t):
        # 予測ホライゾンの長さを計算
        T = self.params.Tf * (1 - np.exp(-self.params.Ta * t))
        # ステップ幅を計算
        h = T / self.params.N
        # 状態の時系列設定
        Xs = np.zeros((self.params.N + 1, 2))
        # 随伴変数の時系列設定
        Ls = np.zeros((self.params.N + 1, 2))
        
        # 状態の初期値を設定
        Xs[0] = X
        
        # 状態方程式を用いて状態の時系列を計算
        for i in range(self.params.N):
            Xs[i + 1] = Xs[i] + h * np.array(self.state_equation(Xs[i], U[i]))
            
        # 終端条件を計算
        Ls[-1,0] = self.params.qxt * (Xs[-1,0] - self.params.x_target)
        Ls[-1,1] = self.params.qvt * Xs[-1,1]
        
        # 随伴方程式を用いて随伴変数の時系列を計算
        for i in range(self.params.N - 1, -1, -1):
            dL = np.array([-self.params.qx * (Xs[i,0] - self.params.x_target) + Ls[i+1,1]/self.params.m * (self.params.k + 3 * self.params.k3 * Xs[i,0]**2),
                        -self.params.qv * Xs[i,1] - Ls[i + 1,0]])
            Ls[i] = Ls[i + 1] - h * dL
        
        # 停留条件式の構成
        F = np.zeros(self.params.N)
        for i in range(self.params.N):
            F[i] = self.params.r * U[i] + Ls[i,1]/self.params.m
            
        return F

    # step 7 GMRES
    def gmres(self, X, t, max_iter:int = 50, trol:float=1e-6, atol:float=1e-8):

        # 変数定義
        self.n = len(self.U)
        self.g = np.zeros(self.n+1) # 右辺の構成
        self.R = np.zeros((self.n, self.n)) # 上三角行列の構成
        self.c = np.zeros(self.n) # Givens回転 c の配列
        self.s = np.zeros(self.n) # Givens回転 s の配列
        self.V = np.zeros((self.n, self.n+1)) # Klynov部分空間の直交基底

        # step 7-1 初期残差の計算
        if self.prev_dU == None:
            self.dU_0 = 0
            self.FudU = 0
        else:
            self.dU_0 = self.prev_dU
            self.FudU = (self.calc_F(self.U + self.params.eps * self.uU_0, X, t) - self.F_u_x_t)/self.params.eps

        self.r0=self.bk - self.FudU

        self.beta = np.linalg.norm(self.r0)
        if self.beta < atol: # 絶対誤差が十分小さい場合
            return self.U
        
        # step 7-2 直交基底 v1 の計算
        v1 = self.r0 / self.beta
        self.V[:,0] = v1

        self.g[0] = self.beta

        max_iter = min(max_iter, self.n)

        for k in range(max_iter):
            # step 7-3 Arnoldi法 step m
            
            #修正グラムシュミット法
            FuVm = (self.calc_F(self.U + self.params.eps * self.V[:,k-1], X, ))


    def calc_next_U(self, X, t):

        # setp 1 時刻 tk の制御対象の状態を取得
        self.X = X

        # step 2 # 制御入力列の更新
        if self.prev_U == None:
            self.U = self.initial_control_input(self.X)
        else:
            self.U = self.prev_U

        # step 3 制御指令の更新
        self.control_command = self.U[0]

        # step 4 dX の計算
        self.dX = self.state_equation(self.X, self.U[0])

        # step 5 停留条件の計算
        self.F_u_x_t = self.calc_F(self.U, self.X, t)

        # step 6 右辺 bkの計算
        self.bk = -(self.calc_F(self.U, self.X + self.params.eps * self.dX, t + self.params.eps) - self.F_u_x_t)/self.params.eps - self.params.zeta * self.F_u_x_t

        # step 7 GMRES の計算

            
        


In [ ]:
def Contenious_GMRES():
    

